# Final Project Analysis

##### Group Member Names: Sarah Wheeler

In [1]:
import pandas as pd
import bqplot

df = pd.read_csv("owid-covid-data.csv")
df.head(5)

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,1/3/2020,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,1/4/2020,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,1/5/2020,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,1/6/2020,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,1/7/2020,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772,NaN,NaN,NaN,NaN


In [2]:
df = df.dropna(subset = ["total_cases", "location"])
df["total_cases"] = pd.to_numeric(df["total_cases"], errors="coerce")

## Code:

 * An interactive dashboard within your Workspace that helps an expert explore your dataset thoroughly.
 * There should be a "dashboard" type aspect to this - i.e. a linked view exploring your dataset in an interactive way (like in Lab \#4) with [bqplot](https://bqplot.github.io/bqplot/).
 * Do not delete any cells, *just comment them out*. Show your work.



In [3]:
xsc2 = bqplot.OrdinalScale()
ysc2 = bqplot.LinearScale()

x_axh = bqplot.Axis(scale = xsc2, label = "Location", side = 'bottom', label_offset = '40')
y_axh = bqplot.Axis(scale = ysc2, label = "Total Cases", orientation = "vertical", side = 'left', label_offset = '-50') #, tick_format='0.2d')
#colax2 = bqplot.ColorAxis(scale = colsc2, side = 'right')

df = df[df["continent"] == "Asia"]
subbed = df[["total_cases", "location"]].groupby("location").max().sort_values("total_cases", ascending=False).head(15)
print(subbed.head(5))

if subbed.empty:
    x_vals = ['']
    y_vals = [0]
else:
    x_vals = subbed.index.tolist()
    y_vals = subbed["total_cases"].tolist()
    print(y_vals)

tooltip = bqplot.Tooltip(fields=["x", "y"], labels=["Location", "Total Cases"])    
bars = bqplot.Bars(x = x_vals, y = y_vals, scales={"x": xsc2, "y": ysc2}, tooltip = tooltip, interactions = {"hover": "tooltip"})
bars.padding = 0.25

fig_bars = bqplot.Figure(marks = [bars], axes = [x_axh, y_axh], title="Top 15 Asian Countries by Total COVID-19 Cases")

fig_bars.fig_margin = {
    'top': 60,
    'bottom': 140,
    'left': 120,
    'right': 60
}

#don't want plot to be squished
fig_bars.layout.width = '1100px'
fig_bars.layout.height = '750px'
fig_bars.layout.min_width = '1100px'
fig_bars.layout.flex = '0 0 auto'

fig_bars

             total_cases
location                
China         99321242.0
India         45003055.0
South Korea   34571873.0
Japan         33803572.0
Turkey        17004677.0
[99321242.0, 45003055.0, 34571873.0, 33803572.0, 17004677.0, 11624000.0, 7624699.0, 6815156.0, 5147359.0, 4841772.0, 4760712.0, 4173631.0, 2771072.0, 2465545.0, 2046133.0]


Figure(axes=[Axis(label='Location', label_offset='40', scale=OrdinalScale(), side='bottom'), Axis(label='Total…

In [4]:
#checking max total_cases for random countries to confirm my aggregation is correct
df['total_cases'][df['location'] == 'Vietnam'].describe()
df['total_cases'][df['location'] == 'South Korea'].describe()
df['total_cases'][df['location'] == 'Kuwait'].describe()

count      1388.000000
mean     431685.515130
std      246200.244523
min          11.000000
25%      169824.500000
50%      477471.500000
75%      662970.000000
max      666551.000000
Name: total_cases, dtype: float64

In [5]:
bars.x

array(['China', 'India', 'South Korea', 'Japan', 'Turkey', 'Vietnam',
       'Iran', 'Indonesia', 'Malaysia', 'Israel', 'Thailand',
       'Philippines', 'Singapore', 'Iraq', 'Bangladesh'], dtype='<U11')

In [6]:
def on_selection(change):
    selected = change['new'] 
    
    if selected and len(selected) == 1:
        i = selected[0]
        print("Selected index:", i)
        print("Location:", x_data[i])
        print("Total Cases:", y_data[i])

In [19]:
from ipywidgets.embed import dependency_state
import json

state = dependency_state([fig_bars])

with open("bar_chart.json", "w") as f:
    json.dump(state, f)

#fig_bars.save("bar_chart.json")
#fig_bars.properties(width='container').save("bar_chart.json")

In [7]:
#fig, total cases by date for Asia
df["date"] = pd.to_datetime(df["date"], errors = 'coerce')
df_time = df.groupby("date", as_index=False)["new_cases"].sum().sort_values("date")

x_data = df_time["date"]
y_data = df_time["new_cases"]

x_sc = bqplot.DateScale()
y_sc = bqplot.LinearScale()

x_ax = bqplot.Axis(scale=x_sc, label="Date")
y_ax = bqplot.Axis(scale=y_sc, label="New COVID-19 Cases", orientation='vertical', label_offset = '-50')

asia_line = bqplot.Lines(
    x=x_data,
    y=y_data,
    scales={"x": x_sc, "y": y_sc},
#    stroke_width=2
    marker="circle",
    marker_size=10
)

tooltip = bqplot.Tooltip(
    fields=["x", "y"],
    labels=["Date", "New Cases"],
    formats=["%Y-%m-%d", ",.0f"]
)

asia_line.tooltip = tooltip
asia_line.interactions = {"hover": "tooltip"}

selected_country_on_asia = bqplot.Lines(x=[],y=[],scales=asia_line.scales,  labels=["Selected Country"], display_legend=True)

fig_asia = bqplot.Figure(marks=[asia_line, selected_country_on_asia], axes=[x_ax, y_ax], title="New COVID-19 Cases Over Time in Asia")
fig_asia.layout.width = "1000px"
fig_asia.layout.height = "500px"

fig_asia

Figure(axes=[Axis(label='Date', scale=DateScale()), Axis(label='New COVID-19 Cases', label_offset='-50', orien…

In [20]:
state = dependency_state([fig_asia])

with open("asia_line.json", "w") as f:
    json.dump(state, f)

In [8]:
#fig_line
import ipywidgets 

country_options = sorted(df["location"].dropna().unique().tolist())

country_dropdown = ipywidgets.Dropdown(
    options=country_options,
    value="China",
    description="Country:"
)

country_df = (
    df[df["location"] == country_dropdown.value][["date", "new_cases"]]
    .sort_values("date")
)

x_sc_country = bqplot.DateScale()
y_sc_country = bqplot.LinearScale()

x_ax_country = bqplot.Axis(scale=x_sc_country, label="Date")
y_ax_country = bqplot.Axis(scale=y_sc_country, label="New Cases", orientation="vertical", label_offset = '-40')

country_line = bqplot.Lines(
    x=country_df["date"],
    y=country_df["new_cases"],
    scales={"x": x_sc_country, "y": y_sc_country},
  #  stroke_width=2,
    marker="circle",
    marker_size= 9
)

country_tooltip = bqplot.Tooltip(
    fields=["x", "y"],
    labels=["Date", "New Cases"],
    formats=["%Y-%m-%d", ",.0f"]
)

country_line.tooltip = country_tooltip
country_line.interactions = {"hover": "tooltip"}

fig_country_time = bqplot.Figure(
    marks=[country_line],
    axes=[x_ax_country, y_ax_country],
    title=f"New COVID-19 Cases Over Time: {country_dropdown.value}"
)
fig_country_time.layout.width = "900px"
fig_country_time.layout.height = "400px"


'''
ch_df = df[df["location"] == "China"]
#vaccine_df["date"] = pd.to_datetime(vaccine_df["date"], errors = 'coerce')

x_data = ch_df["date"]
y_data = ch_df["new_cases"]

x_sc = bqplot.DateScale()
y_sc = bqplot.LinearScale()

x_ax = bqplot.Axis(scale=x_sc, label="Date")
y_ax = bqplot.Axis(scale=y_sc, label="New COVID-19 Cases", orientation='vertical', label_offset = '-40')

line = bqplot.Lines(
    x=x_data,
    y=y_data,
    scales={"x": x_sc, "y": y_sc},
    marker = 'circle',
    marker_size = 5
)

tooltip = bqplot.Tooltip(
    fields=["x", "y"],
    labels=["Date", "New Cases"],
    formats=["%Y-%m-%d", ",.0f"]
)

line.tooltip = tooltip
line.interactions = {"hover": "tooltip"}

fig_line = bqplot.Figure(marks=[line], axes=[x_ax, y_ax])
fig_line.layout.width = "1000px"
fig_line.layout.height = "500px"

display(fig_line)
'''

'\nch_df = df[df["location"] == "China"]\n#vaccine_df["date"] = pd.to_datetime(vaccine_df["date"], errors = \'coerce\')\n\nx_data = ch_df["date"]\ny_data = ch_df["new_cases"]\n\nx_sc = bqplot.DateScale()\ny_sc = bqplot.LinearScale()\n\nx_ax = bqplot.Axis(scale=x_sc, label="Date")\ny_ax = bqplot.Axis(scale=y_sc, label="New COVID-19 Cases", orientation=\'vertical\', label_offset = \'-40\')\n\nline = bqplot.Lines(\n    x=x_data,\n    y=y_data,\n    scales={"x": x_sc, "y": y_sc},\n    marker = \'circle\',\n    marker_size = 5\n)\n\ntooltip = bqplot.Tooltip(\n    fields=["x", "y"],\n    labels=["Date", "New Cases"],\n    formats=["%Y-%m-%d", ",.0f"]\n)\n\nline.tooltip = tooltip\nline.interactions = {"hover": "tooltip"}\n\nfig_line = bqplot.Figure(marks=[line], axes=[x_ax, y_ax])\nfig_line.layout.width = "1000px"\nfig_line.layout.height = "500px"\n\ndisplay(fig_line)\n'

In [9]:
def update_country_chart(change):
    country = change["new"]
    filtered = (
        df[df["location"] == country][["date", "new_cases"]]
        .dropna()
        .sort_values("date")
    )
    country_line.x = filtered["date"]
    country_line.y = filtered["new_cases"]
    fig_country_time.title = f"Daily New COVID-19 Cases: {country}"

country_dropdown.observe(update_country_chart, names="value")

In [10]:
fig_country_time

Figure(axes=[Axis(label='Date', scale=DateScale()), Axis(label='New Cases', label_offset='-40', orientation='v…

In [11]:
#interaction --- bar plot as driver, other plots as "driven"
bars.selected_style = {"opacity": 1.0}
bars.unselected_style = {"opacity": 0.3}
bars.selected = []

def update_country_views(country):
    filtered = (
        df[df["location"] == country][["date", "new_cases"]]
        .dropna()
        .sort_values("date")
    )

    country_line.x = filtered["date"]
    country_line.y = filtered["new_cases"]
    fig_country_time.title = f"Daily New Cases: {country}"

#    selected_country_on_asia = bqplot.Lines(x=[],y=[], scales=asia_line.scales, colors=["orange"],)

    asia_line.labels = ["Asia"]
    asia_line.colors = ["steelblue"]
    asia_line.display_legend = True

    fig_asia.marks = [asia_line, selected_country_on_asia]
    fig_asia.legend_location = "top-left"


    selected_country_on_asia.x = filtered["date"]
    selected_country_on_asia.y = filtered["new_cases"]
    selected_country_on_asia.labels = [country]
    selected_country_on_asia.labels = [country]
    selected_country_on_asia.colors = ["orange"]
    selected_country_on_asia.display_legend = True



def on_bar_select(change):
    selected = change["new"]
    if selected is not None and len(selected) > 0:
        i = selected[0]
        country = subbed.index[i] 
        update_country_views(country)
        
def handle_bar_click(self, event):
    i = event["data"]["index"]
    country = x_vals[i]
    new_opacities = [0.25] * len(x_vals)
    new_opacities[i] = 1.0
    bars.opacities = new_opacities

    filtered = (df[df["location"] == country][["date", "new_cases"]].dropna().sort_values("date"))
    
    selected_country_on_asia.x = filtered["date"]
    selected_country_on_asia.y = filtered["new_cases"]
    selected_country_on_asia.labels = [country]

    country_dropdown.value = country
    
bars.observe(on_bar_select, names="selected")
bars.on_element_click(handle_bar_click)
bars.selected = [0]
update_country_views(subbed.index[0])

In [21]:
state = dependency_state([fig_country_time])

with open("country_line.json", "w") as f:
    json.dump(state, f)

In [12]:
#header = 'COVID-19 Case Dashboard'
import ipywidgets as widgets

#trying to make this top plot un-squished
top_row = ipywidgets.HBox([fig_bars], layout= ipywidgets.Layout(width='125%', justify_content='center', overflow_x='auto'))
controls = widgets.HBox([country_dropdown], layout=widgets.Layout(width='100%', justify_content='flex-end', padding='0 20px 0 0'))
bottom_row = widgets.HBox([fig_asia, fig_country_time], layout=widgets.Layout(width='100%', justify_content='space-around', align_items='flex-start'))

dashboard = widgets.VBox([top_row, controls, bottom_row], layout=widgets.Layout(width='100%', gap='20px', align_items='stretch'))

dashboard

In [13]:
from ipywidgets.embed import dependency_state

state = dependency_state([dashboard])

import json
with open("dashboard.json", "w") as f:
    json.dump(state, f)

## Prose:

* One paragraph explaining how to use the dashboard you created, to help someone who is not an expert understand your dataset.
* A list of 1 or more contextual datasets you have identified, links to where they reside, and a sentence about why they might be useful in telling the final story.
  * by "contextual dataset" here means a dataset that would add context to your chosen dataset. For example, if your dataset is the Champaign bus routes, some interesting contextual datasets could be the Chicago bus routes, or the Springfield bus routes, or the Amtrak routes in Champaign
  * you do not have to do anything with this dataset at the moment beyond writing a bit about why it would be useful. Looking forward, you will want to include "contextual visualizations" (which you may or may not generate on your own) in your Final Project, Part 3 and identifying a possibly useful dataset is a great way to start looking for contextual visualizations.
* If you have identified your dataset as a "large one" (i.e. larger than the GitHub file upload limit) comment on if you want to revise your plan for hosting this data or not. If this does not apply to your dataset please explicitly state this.
* Additionally, please note that as of writing, it is not possible to embed images within Starboard. Be sure to address how you plan on including your contextual dataset to add context to your main dataset given that you won't be able to directly embed images if you plan on using Starboard for Part 3.1 of the Final Project.


### Prose

#### Group Member Names: Sarah Wheeler

The dashboard I created is a continent-specific look into the COVID-19 pandemic. The top plot in the dashboard displays the fifteen countries in Asia with the highest total case counts (through the end of 2023), aiming to provide the viewer with a quick overview of which countries faced the greatest number of affected people. This plot is intended as the driver, so whenever someone clicks one of the bars in the plot, the two plots below are updates as well. To fully understand trends in case count, we must also look at how it is distributed over time. Thus, the second plot (on the left side of the dashboard, below the bar chart) details cases over time for the entirety of Asia. The orange line, which is updated upon a click to the driver plot (the bar plot), represents that countries new cases trend over time in comparison to the continent as a whole. To isolate each country's contribution to the overall aggregation of continental counts, I created the third plot. Viewers can choose from any Asian country (not just the top 15) to view the overall case number at any point throughout the pandemic via the dropdown. 

When all three plots are examined in combination, I intend for experts and the general public alike to be able to explore how this region of the world was affected by the pandemic, and hope that stark differences in case count for countries that are similar can invite discussion on pandemic management and prevention.

The following are contextual datasets I believe would enrich the quality of the dashboard:

         - World Bank Population Dataset (link: https://data.worldbank.org/indicator/SP.POP.TOTL): This dataset tracks the overall population of each country through the dates the COVID-19 data was collected. One can use this data to normalize the total case count variable to better understand what percentage of a country's population the total_cases statistic amounts to. For example, while China has the highest case count of all Asian countries, it also has one of the highest populations. The population variable in the dataset is stagnant and does not update from day-to-day. I believe that using a variable that is not fixed would allow us to perform normalization at specific time points during the pandemic, providing a more holistic oversight.
         
         - Oxford University's COVID-19 Government Response Tracker (link: https://github.com/OxCGRT/covid-policy-dataset) This dataset describes each country's approach to the pandemic, including many of the measures that were taken and when they were implemented. I would be interested in examining the proactivity of each nation's measures, and comparing that to their normalized case count. Ideally, we'd be able to learn about how the speed of measure implementation can affect success, as well as examine which measures overall proved the most effective in preventing case transmission.

The COVID-19 dataset is 92 MB in size, less than GitHub's file upload limit. Thus, I can host my data online on Github, and the alternative hosting options do not apply to my project. Additionally, I do not intend to use Starboard for part 3.1 of the final project, opting for a Jekyll webpage hosted on my Github repository and powered by Jupyter Notebooks.

## Plot Summary

Summarize the characteristics of the dataset in words: what does it represent, what are the fields/columns/rows, what data types are they, etc.

This dataset is comprised of attributes (columns) related to COVID-19, including vaccination rates, test cases, and deaths, aggregated at the country level and collected daily from 2020-2024 (rows). Most variables correspond to float data types as they record amounts of vaccinations, covid cases, mortality, etc. The only columns with an object datatype are the location, continent, and iso_code. I manually changed the date category to datetime to support the creation of my plots. 